## Récupération des noms d'utilisateurs

Ce code scrap r/france pour récupérer des noms d'utilisateurs. 

### Récupération

In [1]:
import requests
import time

URL = "https://api.pullpush.io/reddit/submission/search/"
headers = {
    "User-Agent": "Mozilla/5.0 (Our research; contact: placeholder@example.com)"
}

with open("usernames_raw.txt", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()
    lines = []
    for line in raw_lines:
        lines.append(line.strip('\n'))
    last_line = lines[-1]
    try :
        last_line = float(last_line)
        before = last_line
        users = set(lines[-2::-1])
    except:
        print("No last period saved")
        before = int(time.time())
        users = set(lines)

unique_all = len(users)
users = set()
while unique_all < 100000:
    unique = 0
    while unique < 1000:
        params = {
            "subreddit": "france",
            "size": 100,
            "before": before,
            "sort": "desc"
        }

        try : 
            r = requests.get(URL,headers=headers, params=params, timeout=30)
            r.raise_for_status()
            data = r.json().get("data", [])
        except:
            print('Error from the URL. Waiting.')
            time.sleep(2)

        if not data:
            print("No more data.")
            break

        for post in data:
            user = post.get("author")
            if user and user != "[deleted]":
                users.add(user)
        unique = len(users)

        # store last production date
        before = data[-1]["created_utc"]

        print(f"Unique usernames in batch: {unique}")
        time.sleep(1)
    
    with open("usernames_raw.txt", "a", encoding="utf-8") as f:
        for user in users:
            f.write(f"{user}\n")
        f.write(f'{before}\n')
    
    users = set()
    with open("usernames_raw.txt", "r", encoding="utf-8") as f:
        lines = f.readlines()
        unique_all = len(set(lines))
        print(f'Cleaned usernames set.\nTotal usernames: {unique_all}')


### Nettoyage des noms d'utilisateurs 
On transforme les noms stockés dans `usernames_raw.txt` en un fichier propre `usernames.txt`.

In [ ]:
no_date = set()
with open("usernames_raw.txt", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()
    for line in raw_lines:
        line = line.strip('\n')
        if len(line) >= 10 and line[:10].isdigit():
            line = line[11:]
            
        no_date.add(line)

print(len(no_date))
with open('usernames.txt', 'w', encoding='utf-8') as f:
    for username in no_date:
        f.write(f"{username}\n")

### Alternative en cas d'erreur
En fonction du navigateur et de la connexion, le scraping présenté en cellule 1 peut échouer. Dans ce cas, on utilise la cellule qui suit. 

In [ ]:
import requests
import time
from datetime import datetime, timezone 

def safe_request(url, headers, params, retries=5, backoff=5):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=30)
            r.raise_for_status()
            return r

        except requests.exceptions.SSLError as e:
            print(
                f"SSL error (Cloudflare 525 likely) "
                f"(attempt {attempt}/{retries}): {e}"
            )
            time.sleep(backoff * attempt)

        except requests.exceptions.HTTPError as e:
            status = e.response.status_code if e.response else None
            print(
                f"HTTP error {status} "
                f"(attempt {attempt}/{retries})"
            )
            time.sleep(backoff * attempt)

        except requests.exceptions.RequestException as e:
            print(
                f"Network error "
                f"(attempt {attempt}/{retries}): {e}"
            )
            time.sleep(backoff * attempt)

    return None


URL = "https://api.pullpush.io/reddit/submission/search/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Research project; contact: placeholder@example.com)"
}
FILENAME = "usernames3.txt"
TARGET_UNIQUE = 100000
SUBREDDIT = "france"
PAGE_SIZE = 100
SLEEP_SECONDS = 1

def load_state(filename):
    """
    File format:
    - one username per line
    - last line MUST be a Unix timestamp (int)
    """
    try:
        with open(filename, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        return set(), None
    if not lines:
        return set(), None
    try:
        before = int(lines[-1])
        authors = set(lines[:-1])
        return authors, before
    except ValueError:
        raise RuntimeError(
            "Corrupted state file: last line must be a Unix timestamp"
        )

def save_state(filename, authors, before):
    with open(filename, "w", encoding="utf-8") as f:
        for author in sorted(authors):
            f.write(author + "\n")
        f.write(str(int(before)) + "\n")
 
# ---- INITIAL TIMESTAMP (31 Dec 2025, 23:59 UTC) ----

start_datetime = datetime(2025, 12, 31, 23, 59, tzinfo=timezone.utc)

DEFAULT_BEFORE = int(start_datetime.timestamp())

authors, before = load_state(FILENAME)

if before is None:
    before = DEFAULT_BEFORE
    print("Starting fresh from 2025-12-31 23:59 UTC")

else:
    print(f"Resuming crawl from timestamp {before}")

while len(authors) < TARGET_UNIQUE:
    params = {
        "subreddit": SUBREDDIT,
        "size": PAGE_SIZE,
        "before": before,
        "sort": "desc"
    }

    r = safe_request(URL, HEADERS, params)
    data = r.json().get("data", [])

    if not data:
        print("No more data available.")
        break

    for post in data:
        author = post.get("author")
        if author and author != "[deleted]":
            authors.add(author)

    before = data[-1]["created_utc"]

    save_state(FILENAME, authors, before)

 

    print(f"Unique authors collected: {len(authors)}")

 

    time.sleep(SLEEP_SECONDS)